# 🦠 COVID-19 Data Analysis Project
## Notebook 1: Data Loading & Cleaning

**Author:** Ahmed  
**Date:** March 2026  
**Dataset:** `covid.csv` — 210 countries, 17 features

---

### 📌 Objectives
1. Load the raw dataset from `data/raw/`
2. Inspect structure, data types, and null values
3. Clean and fix data quality issues
4. Engineer new features (fatality rate, recovery rate, etc.)
5. Export the cleaned dataset to `data/processed/`

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print('✅ Libraries imported successfully!')

---
## 📂 Step 2 — Load the Dataset

In [ ]:
# Load the raw data from the data/raw/ folder
df = pd.read_csv('../data/raw/covid.csv')

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
df.head(10)

---
## 🔍 Step 3 — Initial Inspection

In [ ]:
# Column names & data types
print('=== Column Names & Data Types ===')
print(df.dtypes)
print()
print(f'Shape: {df.shape}')

In [ ]:
# Check for missing values
print('=== Missing Values per Column ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False))

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

# Check last rows (often summary/total rows)
print('\n=== Last 5 Rows ===')
df.tail(5)

In [ ]:
# Unique values in categorical columns
print('=== Continents ===')
print(df['Continent'].unique())
print()
print('=== WHO Regions ===')
print(df['WHO Region'].unique())

---
## 🧹 Step 4 — Data Cleaning

In [ ]:
# ── 4.1 Strip whitespace from all string columns ──
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print('✅ Whitespace stripped from column names and string values.')

In [ ]:
# ── 4.2 Rename columns for easier use ──
df.rename(columns={
    'Country/Region'      : 'Country',
    'TotalCases'          : 'TotalCases',
    'NewCases'            : 'NewCases',
    'TotalDeaths'         : 'TotalDeaths',
    'NewDeaths'           : 'NewDeaths',
    'TotalRecovered'      : 'TotalRecovered',
    'NewRecovered'        : 'NewRecovered',
    'ActiveCases'         : 'ActiveCases',
    'Serious,Critical'    : 'SeriousCritical',
    'Tot Cases/1M pop'    : 'CasesPer1M',
    'Deaths/1M pop'       : 'DeathsPer1M',
    'TotalTests'          : 'TotalTests',
    'Tests/1M pop'        : 'TestsPer1M',
    'WHO Region'          : 'WHORegion'
}, inplace=True)

print('✅ Columns renamed.')
print(df.columns.tolist())

In [ ]:
# ── 4.3 Remove invalid/non-country rows ──
# Diamond Princess is a cruise ship, not a country
# Also remove any row where Country is NaN (last blank row)
before = len(df)
df = df[df['Country'].notna()]
df = df[df['Country'] != 'Diamond Princess']
after = len(df)

print(f'✅ Removed {before - after} non-country rows. Remaining: {after} countries.')

In [ ]:
# ── 4.4 Fill missing numeric values ──
# For NewCases, NewDeaths, NewRecovered — missing means 0 new cases reported
df['NewCases']     = df['NewCases'].fillna(0)
df['NewDeaths']    = df['NewDeaths'].fillna(0)
df['NewRecovered'] = df['NewRecovered'].fillna(0)

# For SeriousCritical — fill with 0
df['SeriousCritical'] = df['SeriousCritical'].fillna(0)

# For TotalTests & TestsPer1M — fill with median (some countries didn't report)
df['TotalTests']  = df['TotalTests'].fillna(df['TotalTests'].median())
df['TestsPer1M']  = df['TestsPer1M'].fillna(df['TestsPer1M'].median())

# For TotalRecovered, TotalDeaths, ActiveCases — fill with median
for col in ['TotalRecovered', 'TotalDeaths', 'ActiveCases', 'DeathsPer1M', 'CasesPer1M']:
    df[col] = df[col].fillna(df[col].median())

print('✅ Missing numeric values filled.')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

In [ ]:
# ── 4.5 Fill missing categorical values ──
# WHORegion and Continent — fill with 'Unknown'
df['WHORegion']  = df['WHORegion'].fillna('Unknown')
df['Continent']  = df['Continent'].fillna('Unknown')
df['iso_alpha']  = df['iso_alpha'].fillna('Unknown')

print('✅ Categorical missing values filled with "Unknown".')

In [ ]:
# ── 4.6 Fix data types ──
numeric_cols = [
    'Population', 'TotalCases', 'NewCases', 'TotalDeaths', 'NewDeaths',
    'TotalRecovered', 'NewRecovered', 'ActiveCases', 'SeriousCritical',
    'CasesPer1M', 'DeathsPer1M', 'TotalTests', 'TestsPer1M'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print('✅ Data types fixed to numeric.')
df.dtypes

---
## 🔧 Step 5 — Feature Engineering

We create new calculated columns that are important for analysis:

In [ ]:
# ── 5.1 Case Fatality Rate (CFR) ──
# CFR = (TotalDeaths / TotalCases) × 100
# Measures: what % of confirmed cases resulted in death
df['CaseFatalityRate'] = np.where(
    df['TotalCases'] > 0,
    (df['TotalDeaths'] / df['TotalCases'] * 100).round(2),
    0
)

print('✅ CaseFatalityRate created.')

# ── 5.2 Recovery Rate ──
# RecoveryRate = (TotalRecovered / TotalCases) × 100
df['RecoveryRate'] = np.where(
    df['TotalCases'] > 0,
    (df['TotalRecovered'] / df['TotalCases'] * 100).round(2),
    0
)

print('✅ RecoveryRate created.')

# ── 5.3 Active Case Rate ──
# What % of total cases are still active
df['ActiveCaseRate'] = np.where(
    df['TotalCases'] > 0,
    (df['ActiveCases'] / df['TotalCases'] * 100).round(2),
    0
)

print('✅ ActiveCaseRate created.')

# ── 5.4 Tests per Case ──
# How many tests were done per confirmed case
df['TestsPerCase'] = np.where(
    df['TotalCases'] > 0,
    (df['TotalTests'] / df['TotalCases']).round(2),
    0
)

print('✅ TestsPerCase created.')

---
## ✅ Step 6 — Final Check & Summary

In [ ]:
print('=== Final Dataset Summary ===')
print(f'Shape     : {df.shape}')
print(f'Countries : {df["Country"].nunique()}')
print(f'Continents: {df["Continent"].nunique()} — {df["Continent"].unique().tolist()}')
print(f'Any nulls : {df.isnull().sum().sum()}')
print()
print('=== Columns ===')
print(df.columns.tolist())

In [ ]:
# Quick statistical summary of key metrics
key_cols = ['TotalCases', 'TotalDeaths', 'TotalRecovered', 'ActiveCases',
            'CaseFatalityRate', 'RecoveryRate', 'TestsPerCase']
df[key_cols].describe().round(2)

In [ ]:
# Preview the final cleaned dataset
df.head(10)

---
## 💾 Step 7 — Export Cleaned Data

In [ ]:
# Save cleaned dataset to data/processed/ — this will be used by all subsequent notebooks
df.to_csv('../data/processed/cleaned_covid.csv', index=False)

print('✅ Cleaned dataset saved as: ../data/processed/cleaned_covid.csv')
print(f'   Final shape: {df.shape}')
print(f'   Total countries: {df["Country"].nunique()}')
print()
print('🎉 Notebook 1 Complete! Next: Open 02_exploratory_data_analysis.ipynb')

---

## 📋 Summary of Changes Made

| Step | Action | Detail |
|------|--------|--------|
| 4.1 | Stripped whitespace | Column names + string values |
| 4.2 | Renamed columns | Removed spaces & slashes for easier coding |
| 4.3 | Removed invalid rows | Removed 'Diamond Princess' (cruise ship) + blank row |
| 4.4 | Filled numeric nulls | NewCases/Deaths → 0; Tests → median |
| 4.5 | Filled categorical nulls | WHORegion, Continent → 'Unknown' |
| 4.6 | Fixed data types | All numeric columns → float64 |
| 5.1 | New: CaseFatalityRate | Deaths / Cases × 100 |
| 5.2 | New: RecoveryRate | Recovered / Cases × 100 |
| 5.3 | New: ActiveCaseRate | Active / Cases × 100 |
| 5.4 | New: TestsPerCase | Tests / Cases |